# SMBH Hotspot Dataset Explorer

Interactive EDA for the three processed datasets:
- **i0** (`dpa_dataset_i0.csv`): equatorial, 3229 rows, columns: r, K, a, i, DPA, Period
- **ultradense** (`dpa_dataset_ultradense.csv`): equatorial full-orbit, 6188 rows
- **noneq** (`dpa_dataset_noneq.csv`): non-equatorial, 48001 rows, includes theta

> **Unit note**: r, i, theta, z are stored at 10× physical values in the CSVs (colleague's encoding).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, HBox, VBox

# Repo root
repo_root = Path("/scratch/ralbe/meniar_and_django/smbh_hotspots_repository")
DATA_DIR = repo_root / "data" / "processed"

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("Ready.")

Ready.


## 1. Load datasets

In [2]:
datasets = {
    "i0":         pd.read_csv(DATA_DIR / "dpa_dataset_i0.csv"),
    "ultradense": pd.read_csv(DATA_DIR / "dpa_dataset_ultradense.csv"),
    "noneq":      pd.read_csv(DATA_DIR / "dpa_dataset_noneq.csv"),
}

for name, df in datasets.items():
    print(f"[{name:>10}]  {df.shape[0]:>6} rows × {df.shape[1]:>2} cols  |  {list(df.columns)}")

[        i0]    3229 rows ×  6 cols  |  ['r', 'K', 'a', 'i', 'DPA', 'Period']
[ultradense]    6188 rows × 15 cols  |  ['r', 'K', 'a', 'i', 'Period', 'DPA_0.1', 'DPA_0.2', 'DPA_0.3', 'DPA_0.4', 'DPA_0.5', 'DPA_0.6', 'DPA_0.7', 'DPA_0.8', 'DPA_0.9', 'DPA_1.0']
[     noneq]   48001 rows × 16 cols  |  ['r', 'K', 'a', 'i', 'theta', 'Period', 'DPA_0.1', 'DPA_0.2', 'DPA_0.3', 'DPA_0.4', 'DPA_0.5', 'DPA_0.6', 'DPA_0.7', 'DPA_0.8', 'DPA_0.9', 'DPA_1.0']


## 2. Summary statistics

In [3]:
dataset_widget = widgets.Dropdown(
    options=list(datasets.keys()), value="noneq", description="Dataset:"
)

@interact(dataset=dataset_widget)
def show_stats(dataset):
    df = datasets[dataset]
    stats = df.describe().T
    stats["null"] = df.isnull().sum()
    stats["nunique"] = df.nunique()
    display(stats.style.format("{:.4g}").background_gradient(cmap="Blues", subset=["mean", "std"]))

interactive(children=(Dropdown(description='Dataset:', index=2, options=('i0', 'ultradense', 'noneq'), value='…

## 3. Histograms

In [ ]:
def plot_histograms(dataset_name, cols=None, bins=40, log_y=False):
    df = datasets[dataset_name]
    if cols is None:
        # Default: first 10 columns
        cols = list(df.columns[:10])
    n = len(cols)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.2 * nrows))
    axes = np.array(axes).flatten()
    for ax, col in zip(axes, cols):
        ax.hist(df[col].dropna(), bins=bins, color="steelblue", edgecolor="none", alpha=0.85)
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("count")
        if log_y:
            ax.set_yscale("log")
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(f"Histograms — {dataset_name}", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


ds_sel = widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:")
log_y_sel = widgets.Checkbox(value=False, description="Log y-axis")
bins_sel = widgets.IntSlider(value=40, min=5, max=150, step=5, description="Bins:")

def _update_hist(dataset, bins, log_y):
    df = datasets[dataset]
    # Show only numeric columns (first 12)
    num_cols = list(df.select_dtypes(include=np.number).columns[:12])
    plot_histograms(dataset, cols=num_cols, bins=bins, log_y=log_y)

interact(_update_hist, dataset=ds_sel, bins=bins_sel, log_y=log_y_sel);

interactive(children=(Dropdown(description='Dataset:', index=2, options=('i0', 'ultradense', 'noneq'), value='…

## 4. Column vs column (scatter), coloured by a third column

In [ ]:
def scatter_colored(dataset_name, x_col, y_col, color_col, sample_n=5000, cmap="viridis", alpha=0.5, s=6):
    df = datasets[dataset_name]
    if len(df) > sample_n:
        df = df.sample(sample_n, random_state=0)
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(
        df[x_col], df[y_col],
        c=df[color_col], cmap=cmap,
        alpha=alpha, s=s, linewidths=0
    )
    plt.colorbar(sc, ax=ax, label=color_col)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f"{dataset_name}: {x_col} vs {y_col}  (colour={color_col})")
    plt.tight_layout()
    plt.show()


def _make_scatter_widgets(change=None):
    ds_w = widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:")
    
    def _col_opts(ds):
        return list(datasets[ds].select_dtypes(include=np.number).columns)

    x_w     = widgets.Dropdown(options=_col_opts("noneq"), value="r",      description="X:")
    y_w     = widgets.Dropdown(options=_col_opts("noneq"), value="Period",  description="Y:")
    col_w   = widgets.Dropdown(options=_col_opts("noneq"), value="a",       description="Colour:")
    n_w     = widgets.IntSlider(value=5000, min=100, max=50000, step=500,   description="Sample N:")
    cmap_w  = widgets.Dropdown(
        options=["viridis", "plasma", "inferno", "coolwarm", "RdYlGn", "turbo"],
        value="viridis", description="Colormap:"
    )
    alpha_w = widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05, description="Alpha:")
    s_w     = widgets.IntSlider(value=6, min=1, max=40, step=1,            description="Point size:")

    def on_ds_change(change):
        cols = _col_opts(change["new"])
        x_w.options = y_w.options = col_w.options = cols
        x_w.value   = cols[0] if len(cols) > 0 else None
        y_w.value   = cols[1] if len(cols) > 1 else None
        col_w.value = cols[2] if len(cols) > 2 else None

    ds_w.observe(on_ds_change, names="value")

    ui = VBox([
        HBox([ds_w, x_w, y_w, col_w]),
        HBox([n_w, cmap_w, alpha_w, s_w]),
    ])
    out = widgets.interactive_output(
        scatter_colored,
        dict(dataset_name=ds_w, x_col=x_w, y_col=y_w, color_col=col_w,
             sample_n=n_w, cmap=cmap_w, alpha=alpha_w, s=s_w)
    )
    display(ui, out)

_make_scatter_widgets()

Output()

## 5. DPA time-series profiles

In [6]:
def plot_dpa_profiles(dataset_name, n_samples=30, color_by="a", cmap="plasma", alpha=0.5):
    """Plot DPA_0.1 … DPA_1.0 curves for a random subset of rows."""
    df = datasets[dataset_name]
    dpa_cols = [c for c in df.columns if c.startswith("DPA_")]
    if len(dpa_cols) < 2:
        print(f"Dataset '{dataset_name}' has no DPA time-series columns (only: {[c for c in df.columns if 'DPA' in c]}).")
        return
    orbit_fracs = [float(c.split("_")[1]) for c in dpa_cols]
    sample = df.sample(min(n_samples, len(df)), random_state=42)
    norm_vals = sample[color_by].values
    vmin, vmax = norm_vals.min(), norm_vals.max()
    cmap_fn = plt.get_cmap(cmap)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, row in sample.iterrows():
        color = cmap_fn(norm(row[color_by]))
        ax.plot(orbit_fracs, row[dpa_cols].values, color=color, alpha=alpha, lw=0.9)
    sm = plt.cm.ScalarMappable(cmap=cmap_fn, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label=color_by)
    ax.set_xlabel("Orbit fraction")
    ax.set_ylabel("DPA (deg)")
    ax.set_title(f"DPA profiles — {dataset_name}  (n={len(sample)}, colour={color_by})")
    plt.tight_layout()
    plt.show()


ts_datasets = [k for k, df in datasets.items() if any(c.startswith("DPA_") for c in df.columns)]
ds_ts_w  = widgets.Dropdown(options=ts_datasets, value=ts_datasets[0], description="Dataset:")
n_ts_w   = widgets.IntSlider(value=40, min=5, max=200, step=5, description="# curves:")
cby_ts_w = widgets.Dropdown(options=["a", "i", "theta", "r", "Period"], value="a", description="Colour by:")
cmap_ts_w = widgets.Dropdown(options=["plasma", "viridis", "coolwarm", "turbo"], value="plasma", description="Colormap:")
alpha_ts_w = widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05, description="Alpha:")

def on_ts_ds_change(change):
    df = datasets[change["new"]]
    param_cols = [c for c in df.columns if not c.startswith("DPA_")]
    cby_ts_w.options = param_cols
    cby_ts_w.value = "a" if "a" in param_cols else param_cols[0]
ds_ts_w.observe(on_ts_ds_change, names="value")

ui_ts = VBox([HBox([ds_ts_w, n_ts_w, cby_ts_w, cmap_ts_w, alpha_ts_w])])
out_ts = widgets.interactive_output(
    plot_dpa_profiles,
    dict(dataset_name=ds_ts_w, n_samples=n_ts_w, color_by=cby_ts_w, cmap=cmap_ts_w, alpha=alpha_ts_w)
)
display(ui_ts, out_ts)

Output()

## 6. Correlation heatmap

In [7]:
def plot_corr(dataset_name, method="pearson", max_cols=20):
    df = datasets[dataset_name]
    num_df = df.select_dtypes(include=np.number).iloc[:, :max_cols]
    corr = num_df.corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.55), max(6, len(corr) * 0.5)))
    sns.heatmap(
        corr, mask=mask, annot=len(corr) <= 15, fmt=".2f",
        cmap="RdBu_r", vmin=-1, vmax=1, ax=ax,
        linewidths=0.3, square=True
    )
    ax.set_title(f"{method.capitalize()} correlation — {dataset_name} (first {len(corr)} cols)")
    plt.tight_layout()
    plt.show()

interact(
    plot_corr,
    dataset_name=widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:"),
    method=widgets.Dropdown(options=["pearson", "spearman", "kendall"], value="pearson", description="Method:"),
    max_cols=widgets.IntSlider(value=16, min=3, max=30, step=1, description="Max cols:")
);

interactive(children=(Dropdown(description='Dataset:', index=2, options=('i0', 'ultradense', 'noneq'), value='…

## 7. Pair-plot (target parameters)

In [8]:
def plot_pairplot(dataset_name, color_by, sample_n=3000):
    df = datasets[dataset_name]
    param_cols = [c for c in ["a", "i", "theta", "r", "Period"] if c in df.columns]
    sub = df[param_cols + ([color_by] if color_by not in param_cols else [])].sample(
        min(sample_n, len(df)), random_state=0
    )
    g = sns.pairplot(
        sub, vars=param_cols,
        hue=color_by if sub[color_by].nunique() <= 20 else None,
        plot_kws=dict(alpha=0.3, s=5, linewidths=0),
        diag_kind="hist",
    )
    g.fig.suptitle(f"Pair-plot — {dataset_name}  (colour={color_by}, n={len(sub)})", y=1.01)
    plt.show()

ds_pp_w   = widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:")
cby_pp_w  = widgets.Dropdown(options=["a", "i", "theta", "r"], value="a", description="Colour by:")
n_pp_w    = widgets.IntSlider(value=3000, min=200, max=10000, step=200, description="Sample N:")

def on_pp_ds_change(change):
    df = datasets[change["new"]]
    param_cols = [c for c in ["a", "i", "theta", "r", "Period"] if c in df.columns]
    cby_pp_w.options = param_cols
    cby_pp_w.value = "a" if "a" in param_cols else param_cols[0]
ds_pp_w.observe(on_pp_ds_change, names="value")

ui_pp = VBox([HBox([ds_pp_w, cby_pp_w, n_pp_w])])
out_pp = widgets.interactive_output(plot_pairplot,
    dict(dataset_name=ds_pp_w, color_by=cby_pp_w, sample_n=n_pp_w))
display(ui_pp, out_pp)

Output()

## 8. 1-D marginals: r, a, i, theta (grid coverage)

In [9]:
def plot_coverage(dataset_name):
    df = datasets[dataset_name]
    param_cols = [c for c in ["a", "i", "theta", "r", "K", "Period"] if c in df.columns]
    fig, axes = plt.subplots(1, len(param_cols), figsize=(4 * len(param_cols), 3.5))
    if len(param_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, param_cols):
        vals = df[col].dropna()
        unique = sorted(vals.unique())
        if len(unique) <= 60:
            ax.bar(range(len(unique)), [vals.value_counts()[v] for v in unique],
                   color="steelblue", alpha=0.85)
            ax.set_xticks(range(len(unique)))
            ax.set_xticklabels([f"{v:.2g}" for v in unique], rotation=90, fontsize=7)
        else:
            ax.hist(vals, bins=40, color="steelblue", edgecolor="none", alpha=0.85)
        ax.set_title(col)
        ax.set_ylabel("count")
    fig.suptitle(f"Parameter coverage — {dataset_name}", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(plot_coverage,
    dataset_name=widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:"));

interactive(children=(Dropdown(description='Dataset:', index=2, options=('i0', 'ultradense', 'noneq'), value='…


## 9. DPA mean / std across orbit for each (a, r) combination

In [10]:
def plot_dpa_stats(dataset_name, stat="mean", color_by="r"):
    df = datasets[dataset_name]
    dpa_cols = [c for c in df.columns if c.startswith("DPA_")]
    if not dpa_cols:
        print("No DPA_ columns.")
        return
    orbit_fracs = np.array([float(c.split("_")[1]) for c in dpa_cols])
    dpa_vals = df[dpa_cols].values
    if stat == "mean":
        agg = dpa_vals.mean(axis=1)
        label = "Mean DPA across orbit (deg)"
    elif stat == "std":
        agg = dpa_vals.std(axis=1)
        label = "Std DPA across orbit (deg)"
    elif stat == "range":
        agg = dpa_vals.max(axis=1) - dpa_vals.min(axis=1)
        label = "Range DPA across orbit (deg)"

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    # Left: agg vs spin 'a'
    if "a" in df.columns:
        sc0 = axes[0].scatter(df["a"], agg, c=df[color_by] if color_by in df.columns else "steelblue",
                              cmap="plasma", alpha=0.3, s=5, linewidths=0)
        axes[0].set_xlabel("spin  a")
        axes[0].set_ylabel(label)
        if color_by in df.columns:
            plt.colorbar(sc0, ax=axes[0], label=color_by)
    # Right: agg histogram
    axes[1].hist(agg, bins=60, color="steelblue", edgecolor="none", alpha=0.85)
    axes[1].set_xlabel(label)
    axes[1].set_ylabel("count")
    fig.suptitle(f"DPA {stat} — {dataset_name}", fontsize=13)
    plt.tight_layout()
    plt.show()

ts_dsets = [k for k, df in datasets.items() if any(c.startswith("DPA_") for c in df.columns)]
interact(
    plot_dpa_stats,
    dataset_name=widgets.Dropdown(options=ts_dsets, value=ts_dsets[0], description="Dataset:"),
    stat=widgets.Dropdown(options=["mean", "std", "range"], value="std", description="Stat:"),
    color_by=widgets.Dropdown(options=["r", "a", "i", "theta", "Period"], value="r", description="Colour by:")
);

interactive(children=(Dropdown(description='Dataset:', options=('ultradense', 'noneq'), value='ultradense'), D…

## 10. Filter & inspect rows

In [11]:
def show_filtered(dataset_name, filter_col, vmin, vmax, n_show=20):
    df = datasets[dataset_name]
    if filter_col not in df.columns:
        print(f"Column '{filter_col}' not in {dataset_name}.")
        return
    filtered = df[(df[filter_col] >= vmin) & (df[filter_col] <= vmax)]
    print(f"{len(filtered)} / {len(df)} rows match  {filter_col} ∈ [{vmin:.3g}, {vmax:.3g}]")
    display(filtered.head(n_show))

ds_f_w  = widgets.Dropdown(options=list(datasets.keys()), value="noneq", description="Dataset:")
col_f_w = widgets.Text(value="a", description="Filter col:")
vmin_w  = widgets.FloatText(value=-1.0, description="Min:")
vmax_w  = widgets.FloatText(value=1.0,  description="Max:")
nshow_w = widgets.IntSlider(value=10, min=5, max=100, step=5, description="Show rows:")

ui_f  = VBox([HBox([ds_f_w, col_f_w, vmin_w, vmax_w, nshow_w])])
out_f = widgets.interactive_output(
    show_filtered,
    dict(dataset_name=ds_f_w, filter_col=col_f_w, vmin=vmin_w, vmax=vmax_w, n_show=nshow_w)
)
display(ui_f, out_f)

Output()